# Delete data from database (influxdb)

---
**Notebook version**: `1` (16 Jul 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ⚠️ About this notebook — destructive!
This notebook **permanently deletes** data from the InfluxDB database using diive's in-house engine ([`InfluxIO.delete`](../diive/core/io/db/influx/influxio.py), in `diive/core/io/db/influx`, needs `uv sync --group db`). There is no undo.

For safety, **every `dbc.delete(...)` call below is commented out**. Set the parameters for exactly what you want to remove, double-check them, then uncomment the single call you intend to run. Nothing is deleted on a top-to-bottom *Run All*.

Deletion is scoped by `bucket` + `measurements` + `fields` + `data_version` over the `[START, STOP)` time range. A different `data_version` (e.g. `raw`) and a different bucket (e.g. `{SITE}_raw`) are never touched by a delete aimed at processed data.

## ⏱️ Timestamp convention
The database stores timestamps in **UTC**. `START` and `STOP` below are interpreted in the timezone given by `TIMEZONE_OFFSET_TO_UTC_HOURS` (e.g. `1` for CET winter time) and converted to UTC for the delete. `START` **is** included; `STOP` is the upper bound and **is not** included.

## Imports

In [1]:
import importlib.metadata
import warnings
from datetime import datetime

from diive.core.io.db.influx import InfluxIO  # diive's in-house InfluxDB engine (needs: uv sync --group db)

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
print(f"Last run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

Last run: 2026-07-30 13:42:45
diive v0.91.0


## ✏️ Config folder

In [2]:
DIRCONF = r'F:\dev\poet\configs'  # <-- set to your config folder
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

## 🔌 Connect to database

In [3]:
dbc = InfluxIO(dirconf=DIRCONF)

> Reading configuration files was successful.

v Connection to database works.

Optional — print the full `delete` docstring (all parameters and targeting examples):

In [4]:
# help(dbc.delete)

## 🎯 How targeting works
`measurements` and `fields` each accept a **list** or **`True`** (= all):

| Goal | `measurements` | `fields` |
|---|---|---|
| Specific variables in specific measurements | `['TA', 'SW']` | `['TA_T1_1_1', 'SW_T1_1_1']` |
| All variables of a measurement | `['TA']` | `True` |
| Specific variables across all measurements | `True` | `['TA_T1_1_1']` |
| Everything of a data version | `True` | `True` |

Every deletion is additionally scoped to one `data_version` and the `[START, STOP)` range.

## Delete specific variables
Removes only the named `FIELDS` in the named `MEASUREMENTS`, for the given `DATA_VERSION` and time range.

> ⚠️ Destructive. Uncomment the `dbc.delete(...)` call to run it.

In [5]:
# BUCKET = 'ch-aws_processed'
# DATA_VERSION = 'meteoscreening_diive'
# MEASUREMENTS = ['LW']
# FIELDS = ['LW_BC_IN_T1_2_1', 'LW_BC_OUT_T1_2_1']
# START = '2021-05-05 00:00:01'  # included
# STOP = '2023-11-29 00:00:01'   # not included
# TIMEZONE_OFFSET_TO_UTC_HOURS = 1
#
# # Uncomment to permanently delete the variables above:
# # dbc.delete(
# #     bucket=BUCKET,
# #     measurements=MEASUREMENTS,
# #     fields=FIELDS,
# #     start=START,
# #     stop=STOP,
# #     timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
# #     data_version=DATA_VERSION,
# # )

## Delete all data of a specific data version
`MEASUREMENTS = True` and `FIELDS = True` remove **every variable in every measurement** for the given `DATA_VERSION` and time range. Use a wide time range to catch everything.

> ⚠️ Very destructive — this wipes an entire data version. Uncomment the `dbc.delete(...)` call to run it.

In [ ]:
BUCKET = 'ch-aws_processed'
DATA_VERSION = 'fluxnet_v2026'
MEASUREMENTS = True  # True = all measurements
FIELDS = True        # True = all fields
START = '1995-01-01 00:00:01'  # included
STOP = '2027-01-01 00:00:01'   # not included
TIMEZONE_OFFSET_TO_UTC_HOURS = 1

# Uncomment to permanently delete the ENTIRE data version above:
# dbc.delete(
#     bucket=BUCKET,
#     measurements=MEASUREMENTS,
#     fields=FIELDS,
#     start=START,
#     stop=STOP,
#     timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
#     data_version=DATA_VERSION,
# )

## 🔎 Check what is in the bucket (read-only)
Run this before and after a delete: the data version disappears from the list once all of its records are gone. Nothing is deleted here.

The `measurements=True` lookup used by `delete` is scoped to `DATA_VERSION` and searches the full history, so a version whose newest record is older than 30 days is found as well. If the version is not in the bucket, `delete` raises instead of reporting that it deleted nothing.

In [7]:
print(f"Data versions in {BUCKET}:")
print(dbc.show_data_versions_in_bucket(bucket=BUCKET, verbose=False))

print(f"\nMeasurements in {BUCKET} holding data of {DATA_VERSION}:")
print(dbc.show_measurements_in_bucket(bucket=BUCKET, data_version=DATA_VERSION, verbose=False))

Data versions in ch-aws_processed:
['eddypro_level-0', 'fluxnet_v2024', 'fluxnet_v2026', 'fluxnet_ww2020', 'meteoscreening_diive', 'meteoscreening_mst']

Measurements in ch-aws_processed holding data of fluxnet_v2026:
['CO2', 'G', 'H', 'LE', 'LW', 'PA', 'PPFD', 'PREC', 'RH', 'SW', 'SWC', 'TA', 'TS', 'TURBULENCE', 'VPD', 'WIND']


## ✅ End of notebook

In [8]:
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Finished: 2026-07-30 13:43:02
